# AutoDDG Pipeline — Scalable Dataset Description Generation
**Team DataScribes** | Rishabh Patil 

This notebook implements an end-to-end big data pipeline for automatically generating high-quality textual descriptions for open datasets using **Apache Spark** on the NYU HPC DataProc cluster and **GPT-4o-mini** via the OpenAI API.

## Pipeline Overview
| Step | Description |
|------|-------------|
| 1 | Configuration & Environment Setup |
| 2 | Library Imports |
| 3 | Spark Session Initialization |
| 4 | Step 1 — Metadata Ingestion |
| 5 | Step 2 — CSV Sample Fetching |
| 6 | Step 3 — Data Preparation |
| 7 | Step 4 — LLM Description Generation |
| 8 | Step 5 — Evaluation Table |
| 9 | Evaluation — Text Similarity Metrics |
| 10 | Evaluation — Retrieval (NDCG@10) |
| 11 | Evaluation — BERTScore |
| 12 | Qualitative Evaluation |

---
## Step 1 — Configuration

All pipeline parameters are defined here. Edit these values before running the notebook.

| Parameter | Value | Description |
|-----------|-------|-------------|
| `NYC_LIMIT` | 200 | Datasets to fetch from NYC Open Data |
| `DATA_GOV_LIMIT` | 200 | Datasets to fetch from Data.gov |
| `MODEL_NAME` | gpt-4o-mini | LLM used for description generation |
| `MAX_SAMPLE_ROWS` | 20 | CSV rows sampled per dataset |
| `REQUEST_TIMEOUT` | 40s | HTTP timeout for CSV downloads |
| `MAX_ELAPSED_SECONDS` | 60s | Max streaming time per dataset |
| `MAX_CHARS` | 12000 | Max characters sent to LLM |
| `MAX_LINES` | 20 | Max lines sent to LLM |

All intermediate outputs are stored in HDFS under the user's directory on the DataProc cluster.

In [1]:
NYC_LIMIT       = 200   # datasets from NYC Open Data
DATA_GOV_LIMIT  = 200    # datasets from Data.gov

HDFS_BASE = "hdfs:///user/rbp5812_nyu_edu/pipeline"
LOCAL_OUTPUT_PARQUET = "/home/rbp5812_nyu_edu/outputs/descriptions.parquet"

MODEL_NAME          = "gpt-4o-mini"

MAX_SAMPLE_ROWS     = 20
REQUEST_TIMEOUT     = 40
MAX_ELAPSED_SECONDS = 60
MAX_CHARS           = 12000  # was 8000
MAX_LINES           = 20  

# Derived HDFS paths
HDFS_METADATA = f"{HDFS_BASE}/step1_metadata"
HDFS_SAMPLES  = f"{HDFS_BASE}/step2_samples"
HDFS_PREPARED = f"{HDFS_BASE}/step3_prepared"
HDFS_EVAL     = f"{HDFS_BASE}/step5_evaluation"

print(f"NYC datasets      : {NYC_LIMIT}")
print(f"Data.gov datasets : {DATA_GOV_LIMIT}")
print(f"Total requested   : {NYC_LIMIT + DATA_GOV_LIMIT}")
print(f"Model             : {MODEL_NAME}")
print(f"HDFS base         : {HDFS_BASE}")
print(f"Local output      : {LOCAL_OUTPUT_PARQUET}")


NYC datasets      : 200
Data.gov datasets : 200
Total requested   : 400
Model             : gpt-4o-mini
HDFS base         : hdfs:///user/rbp5812_nyu_edu/pipeline
Local output      : /home/rbp5812_nyu_edu/outputs/descriptions.parquet


---
## Step 2 — Library Imports

We import standard Python libraries alongside:
- **`pandas`** — local DataFrame manipulation
- **`requests`** — HTTP calls to open data APIs
- **`openai`** — GPT-4o-mini API client
- **`pyspark`** — distributed processing on the DataProc cluster
- **`autoddg`** — the AutoDDG framework for description generation

In [2]:
from __future__ import annotations

import csv
import io
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests
from openai import OpenAI
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from autoddg import AutoDDG

csv.field_size_limit(sys.maxsize)
print("Imports OK.")


Imports OK.


---
## Step 3 — Spark Session Initialization

We initialize a Spark session connected to the **YARN** cluster manager on NYU DataProc. The `--deploy-mode client` flag ensures the driver runs on the master node, which is required for interactive Jupyter execution.

The cluster consists of:
- 1 master node (`nyu-dataproc-m`) — n1-standard-32
- 2 worker nodes (`nyu-dataproc-w-0`, `nyu-dataproc-w-1`) — n1-standard-16 each
- 1 preemptible secondary worker — n1-standard-16

In [3]:
import os

# Remove conflicting deploy mode setting
os.environ.pop("SPARK_SUBMIT_OPTS", None)
os.environ["PYSPARK_SUBMIT_ARGS"] = "--master yarn --deploy-mode client pyspark-shell"

spark = (
    SparkSession.builder
    .appName("autoddg_pipeline")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .getOrCreate()
)
print(spark)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/02 21:43:39 INFO SparkEnv: Registering MapOutputTracker
26/05/02 21:43:40 INFO SparkEnv: Registering BlockManagerMaster
26/05/02 21:43:40 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/05/02 21:43:40 INFO SparkEnv: Registering OutputCommitCoordinator


---
## Step 4 — Metadata Ingestion

### Data Sources
We collect dataset metadata from two open data repositories:

**NYC Open Data** (Socrata API)
- Catalog endpoint returns a list of all datasets
- For each dataset we fetch detailed metadata including column names, types, and license info
- Download URLs are constructed using the Socrata CSV export format

**Data.gov** (CKAN Catalog API)
- Search endpoint returns dataset records in DCAT format
- We extract title, description, keywords, and CSV download links from the `dcat.distribution` field
- Many Data.gov datasets do not have direct CSV links — these are handled gracefully

### Schema
All metadata is normalized into a unified Spark schema with 14 fields and written to HDFS as Parquet for efficient downstream processing.

In [4]:
NYC_CATALOG_URL     = "https://data.cityofnewyork.us/api/views.json"
NYC_VIEW_URL_TMPL   = "https://data.cityofnewyork.us/api/views/{dataset_id}.json"
DATA_GOV_SEARCH_URL = "https://catalog.data.gov/search"


def safe_get(dct: Dict[str, Any], key: str, default=None):
    return dct[key] if key in dct else default


def fetch_json(
    url: str,
    params: Optional[Dict[str, Any]] = None,
    retries: int = 3,
    sleep_seconds: float = 2.0,
) -> Any:
    last_error: Optional[Exception] = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(
                url, params=params, timeout=60,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()
            return response.json()
        except Exception as e:
            last_error = e
            print(f"[WARN] fetch failed attempt={attempt}/{retries} url={url}: {e}")
            if attempt < retries:
                time.sleep(sleep_seconds)
    raise RuntimeError(f"Failed to fetch URL after {retries} attempts: {url}") from last_error


def extract_keywords(category: Optional[str], tags: Optional[List[str]]) -> List[str]:
    out: List[str] = []
    if category:
        out.append(str(category))
    if tags:
        out.extend([str(t) for t in tags if t is not None])
    seen = set()
    deduped: List[str] = []
    for x in out:
        if x not in seen:
            seen.add(x)
            deduped.append(x)
    return deduped


def extract_column_names(columns: Optional[List[Dict[str, Any]]]) -> List[str]:
    names: List[str] = []
    for col in columns or []:
        name = safe_get(col, "name")
        if name:
            names.append(str(name))
    return names


def extract_column_types(columns: Optional[List[Dict[str, Any]]]) -> List[str]:
    types_: List[str] = []
    for col in columns or []:
        dtype = safe_get(col, "dataTypeName")
        types_.append(str(dtype) if dtype is not None else "unknown")
    return types_


def normalize_nyc_dataset(summary_entry: Dict[str, Any], detail_entry: Dict[str, Any]) -> Dict[str, Any]:
    dataset_id      = safe_get(summary_entry, "id")
    title           = safe_get(summary_entry, "name")
    description     = safe_get(summary_entry, "description")
    category        = safe_get(summary_entry, "category")
    tags            = safe_get(summary_entry, "tags", [])
    rows_updated_at = safe_get(summary_entry, "rowsUpdatedAt")
    detail_columns  = safe_get(detail_entry, "columns", [])
    license_info    = safe_get(detail_entry, "license")
    license_name    = None
    if isinstance(license_info, dict):
        license_name = safe_get(license_info, "name")
    return {
        "dataset_id":            str(dataset_id) if dataset_id is not None else None,
        "source":                "nyc_open_data",
        "title":                 str(title) if title is not None else None,
        "original_description":  str(description) if description is not None else None,
        "keywords_json":         json.dumps(extract_keywords(category, tags), ensure_ascii=False),
        "column_names_json":     json.dumps(extract_column_names(detail_columns), ensure_ascii=False),
        "column_types_raw_json": json.dumps(extract_column_types(detail_columns), ensure_ascii=False),
        "download_url":          f"https://data.cityofnewyork.us/api/views/{dataset_id}/rows.csv?accessType=DOWNLOAD" if dataset_id else None,
        "landing_page_url":      f"https://data.cityofnewyork.us/d/{dataset_id}" if dataset_id else None,
        "record_count_estimate": None,
        "last_updated":          str(rows_updated_at) if rows_updated_at is not None else None,
        "license":               str(license_name) if license_name is not None else None,
        "sample_rows_json":      None,
        "raw_metadata_json":     json.dumps(
            {"summary_entry": summary_entry, "detail_entry": detail_entry},
            ensure_ascii=False),
    }


def fetch_nyc_metadata(limit: int) -> List[Dict[str, Any]]:
    payload = fetch_json(NYC_CATALOG_URL)
    if not isinstance(payload, list):
        raise ValueError("Expected NYC catalog response to be a list.")
    trimmed = payload[:limit]
    rows: List[Dict[str, Any]] = []
    for idx, entry in enumerate(trimmed, start=1):
        dataset_id = safe_get(entry, "id")
        if not dataset_id:
            continue
        print(f"[INFO] NYC detail {idx}/{len(trimmed)} dataset_id={dataset_id}")
        detail = fetch_json(NYC_VIEW_URL_TMPL.format(dataset_id=dataset_id))
        rows.append(normalize_nyc_dataset(entry, detail))
    return rows


def normalize_data_gov_dataset(entry: Dict[str, Any]) -> Dict[str, Any]:
    dcat         = entry.get("dcat", {}) or {}
    dataset_id   = entry.get("identifier") or dcat.get("identifier")
    title        = entry.get("title") or dcat.get("title")
    notes        = entry.get("description") or dcat.get("description")
    tags         = entry.get("keyword") or dcat.get("keyword") or []
    modified     = dcat.get("modified")
    license_url  = dcat.get("license")
    landing_page = dcat.get("landingPage")
    distributions = dcat.get("distribution", []) or []
    csv_url = None
    for dist in distributions:
        if not isinstance(dist, dict):
            continue
        access_url = dist.get("accessURL") or dist.get("downloadURL")
        media_type = (dist.get("mediaType") or "").lower()
        fmt        = (dist.get("format") or "").lower()
        if access_url and (
            "csv" in media_type or "csv" in fmt
            or str(access_url).lower().endswith(".csv")
        ):
            csv_url = access_url
            break
    return {
        "dataset_id":            str(dataset_id) if dataset_id is not None else None,
        "source":                "data_gov",
        "title":                 str(title) if title is not None else None,
        "original_description":  str(notes) if notes is not None else None,
        "keywords_json":         json.dumps(
            [str(x) for x in tags] if isinstance(tags, list) else [], ensure_ascii=False),
        "column_names_json":     json.dumps([], ensure_ascii=False),
        "column_types_raw_json": json.dumps([], ensure_ascii=False),
        "download_url":          csv_url,
        "landing_page_url":      landing_page,
        "record_count_estimate": None,
        "last_updated":          str(modified) if modified is not None else None,
        "license":               str(license_url) if license_url is not None else None,
        "sample_rows_json":      None,
        "raw_metadata_json":     json.dumps(entry, ensure_ascii=False),
    }


def fetch_data_gov_metadata(limit: int) -> List[Dict[str, Any]]:
    params  = {"q": "", "per_page": limit}
    payload = fetch_json(DATA_GOV_SEARCH_URL, params=params)
    if not isinstance(payload, dict) or "results" not in payload:
        raise ValueError("Expected Data.gov search response with a 'results' key.")
    results = payload["results"]
    if not isinstance(results, list):
        raise ValueError("Expected Data.gov 'results' to be a list.")
    return [normalize_data_gov_dataset(entry) for entry in results[:limit]]


def build_metadata_schema() -> T.StructType:
    return T.StructType([
        T.StructField("dataset_id",            T.StringType(), True),
        T.StructField("source",                T.StringType(), True),
        T.StructField("title",                 T.StringType(), True),
        T.StructField("original_description",  T.StringType(), True),
        T.StructField("keywords_json",         T.StringType(), True),
        T.StructField("column_names_json",     T.StringType(), True),
        T.StructField("column_types_raw_json", T.StringType(), True),
        T.StructField("download_url",          T.StringType(), True),
        T.StructField("landing_page_url",      T.StringType(), True),
        T.StructField("record_count_estimate", T.LongType(),   True),
        T.StructField("last_updated",          T.StringType(), True),
        T.StructField("license",               T.StringType(), True),
        T.StructField("sample_rows_json",      T.StringType(), True),
        T.StructField("raw_metadata_json",     T.StringType(), True),
    ])


### Fetching Metadata
We now call both APIs and combine the results into a single Spark DataFrame, then write it to HDFS at `step1_metadata`.

In [5]:
print(f"[INFO] Fetching NYC metadata: {NYC_LIMIT}")
nyc_rows = fetch_nyc_metadata(limit=NYC_LIMIT)

print(f"[INFO] Fetching Data.gov metadata: {DATA_GOV_LIMIT}")
data_gov_rows = fetch_data_gov_metadata(limit=DATA_GOV_LIMIT)

all_rows = nyc_rows + data_gov_rows
if not all_rows:
    raise ValueError("No metadata rows were fetched from either source.")

metadata_df = spark.createDataFrame(all_rows, schema=build_metadata_schema())

print(f"[INFO] Writing {metadata_df.count()} combined rows to {HDFS_METADATA}")
metadata_df.write.mode("overwrite").parquet(HDFS_METADATA)

metadata_df.groupBy("source").count().show(truncate=False)
metadata_df.select("dataset_id", "source", "title", "download_url").show(30, truncate=False)


[INFO] Fetching NYC metadata: 200
[INFO] NYC detail 1/200 dataset_id=qhkz-4dqm
[INFO] NYC detail 2/200 dataset_id=wgnh-qwsg
[INFO] NYC detail 3/200 dataset_id=naav-ygga
[INFO] NYC detail 4/200 dataset_id=5mb3-padx
[INFO] NYC detail 5/200 dataset_id=i2im-iqtt
[INFO] NYC detail 6/200 dataset_id=gdk4-mbsv
[INFO] NYC detail 7/200 dataset_id=pztn-9bne
[INFO] NYC detail 8/200 dataset_id=5ucz-vwe8
[INFO] NYC detail 9/200 dataset_id=m5vz-tzqv
[INFO] NYC detail 10/200 dataset_id=8zf9-spf8
[INFO] NYC detail 11/200 dataset_id=wh8n-imgd
[INFO] NYC detail 12/200 dataset_id=ct66-47at
[INFO] NYC detail 13/200 dataset_id=6up2-gnw8
[INFO] NYC detail 14/200 dataset_id=76ig-c548
[INFO] NYC detail 15/200 dataset_id=rixx-fc37
[INFO] NYC detail 16/200 dataset_id=aq7i-eu5q
[INFO] NYC detail 17/200 dataset_id=ag7h-2pg6
[INFO] NYC detail 18/200 dataset_id=kb2e-tjy3
[INFO] NYC detail 19/200 dataset_id=6ztr-wgff
[INFO] NYC detail 20/200 dataset_id=vhqf-adkz
[INFO] NYC detail 21/200 dataset_id=kbgp-72qi
[INFO] NY

[INFO] NYC detail 178/200 dataset_id=tdej-swyi
[INFO] NYC detail 179/200 dataset_id=k462-uqyk
[INFO] NYC detail 180/200 dataset_id=bzg2-2abf
[INFO] NYC detail 181/200 dataset_id=592z-n7dk
[INFO] NYC detail 182/200 dataset_id=2ync-kihj
[INFO] NYC detail 183/200 dataset_id=2er2-jqsx
[INFO] NYC detail 184/200 dataset_id=gtdx-4w36
[INFO] NYC detail 185/200 dataset_id=n47m-7kn5
[INFO] NYC detail 186/200 dataset_id=gq2c-wem9
[INFO] NYC detail 187/200 dataset_id=j62s-m9yf
[INFO] NYC detail 188/200 dataset_id=shr7-eqdc
[INFO] NYC detail 189/200 dataset_id=fcnc-95cn
[INFO] NYC detail 190/200 dataset_id=tmt9-43em
[INFO] NYC detail 191/200 dataset_id=53jq-yvwd
[INFO] NYC detail 192/200 dataset_id=sm2x-35i7
[INFO] NYC detail 193/200 dataset_id=mtj6-vmci
[INFO] NYC detail 194/200 dataset_id=6ax4-q5k4
[INFO] NYC detail 195/200 dataset_id=hjz2-y62k
[INFO] NYC detail 196/200 dataset_id=fkec-mjr6
[INFO] NYC detail 197/200 dataset_id=sstf-x9es
[INFO] NYC detail 198/200 dataset_id=p36g-evi5
[INFO] NYC de

26/05/02 21:49:01 WARN TaskSetManager: Stage 0 contains a task of very large size (4665 KiB). The maximum recommended task size is 1000 KiB.


[INFO] Writing 400 combined rows to hdfs:///user/rbp5812_nyu_edu/pipeline/step1_metadata


26/05/02 21:49:05 WARN TaskSetManager: Stage 3 contains a task of very large size (4665 KiB). The maximum recommended task size is 1000 KiB.
26/05/02 21:49:09 WARN TaskSetManager: Stage 4 contains a task of very large size (4665 KiB). The maximum recommended task size is 1000 KiB.


+-------------+-----+
|source       |count|
+-------------+-----+
|nyc_open_data|200  |
|data_gov     |200  |
+-------------+-----+



26/05/02 21:49:11 WARN TaskSetManager: Stage 7 contains a task of very large size (4665 KiB). The maximum recommended task size is 1000 KiB.


+----------+-------------+---------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------+
|dataset_id|source       |title                                                                                              |download_url                                                                  |
+----------+-------------+---------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------+
|qhkz-4dqm |nyc_open_data|Citywide Mobility Survey - Vehicle 2024                                                            |https://data.cityofnewyork.us/api/views/qhkz-4dqm/rows.csv?accessType=DOWNLOAD|
|wgnh-qwsg |nyc_open_data|Citywide Mobility Survey - Trip 2024                                                               |https://data.cityofnewyork.us/api/views/wgnh-qwsg/

### Metadata Summary
Displaying dataset counts per source and a preview of the ingested metadata.

In [6]:
metadata_df.groupBy("source").count().show(truncate=False)
metadata_df.select("dataset_id", "source", "title", "download_url").show(30, truncate=False)

26/05/02 21:49:12 WARN TaskSetManager: Stage 8 contains a task of very large size (4665 KiB). The maximum recommended task size is 1000 KiB.
26/05/02 21:49:13 WARN TaskSetManager: Stage 11 contains a task of very large size (4665 KiB). The maximum recommended task size is 1000 KiB.


+-------------+-----+
|source       |count|
+-------------+-----+
|data_gov     |200  |
|nyc_open_data|200  |
+-------------+-----+

+----------+-------------+---------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------+
|dataset_id|source       |title                                                                                              |download_url                                                                  |
+----------+-------------+---------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------+
|qhkz-4dqm |nyc_open_data|Citywide Mobility Survey - Vehicle 2024                                                            |https://data.cityofnewyork.us/api/views/qhkz-4dqm/rows.csv?accessType=DOWNLOAD|
|wgnh-qwsg |nyc_open_data|Citywide Mobility

---
## Step 5 — CSV Sample Fetching

For each dataset that has a valid download URL, we stream the first `MAX_SAMPLE_ROWS` rows of the CSV file. This avoids downloading entire datasets which can be gigabytes in size.

**Key design decisions:**
- **Streaming**: We use `response.iter_lines()` to read line by line, stopping after enough rows
- **Timeout guard**: A wall-clock timer stops fetching if `MAX_ELAPSED_SECONDS` is exceeded
- **Retry logic**: Up to 3 attempts per dataset with 2-second backoff
- **Graceful failure**: Datasets that fail (missing URL, 403, timeout) are marked as `error` and excluded from later steps — the pipeline continues

Results are written to HDFS at `step2_samples`.

In [7]:
def fetch_csv_sample_text(
    download_url: str,
    max_rows: int = 5,
    retries: int = 3,
    sleep_seconds: float = 2.0,
    request_timeout: int = 20,
    max_elapsed_seconds: int = 30,
) -> Optional[str]:
    last_error: Optional[Exception] = None
    for attempt in range(1, retries + 1):
        start_time = time.time()
        try:
            with requests.get(
                download_url,
                timeout=request_timeout,
                stream=True,
                headers={"User-Agent": "Mozilla/5.0"},
            ) as response:
                response.raise_for_status()
                lines: List[str] = []
                for line in response.iter_lines(decode_unicode=True):
                    if time.time() - start_time > max_elapsed_seconds:
                        raise TimeoutError(f"sample fetch exceeded {max_elapsed_seconds}s")
                    if line is None:
                        continue
                    line = line.strip()
                    if not line:
                        continue
                    lines.append(line)
                    if len(lines) >= max_rows + 1:
                        break
                if not lines:
                    return None
                reader    = csv.reader(io.StringIO("\n".join(lines)))
                rows      = list(reader)
                if not rows:
                    return None
                header    = rows[0]
                data_rows = rows[1:]
                output    = io.StringIO()
                writer    = csv.writer(output, lineterminator="\n")
                writer.writerow(header)
                writer.writerows(data_rows)
                sample_csv = output.getvalue().strip()
                return sample_csv if sample_csv else None
        except Exception as e:
            last_error = e
            print(f"[WARN] sample fetch failed attempt={attempt}/{retries} url={download_url}: {e}")
            if attempt < retries:
                time.sleep(sleep_seconds)
    print(f"[ERROR] giving up on url={download_url}: {last_error}")
    return None


def build_samples_schema() -> T.StructType:
    return T.StructType([
        T.StructField("dataset_id",           T.StringType(), True),
        T.StructField("source",               T.StringType(), True),
        T.StructField("title",                T.StringType(), True),
        T.StructField("original_description", T.StringType(), True),
        T.StructField("download_url",         T.StringType(), True),
        T.StructField("landing_page_url",     T.StringType(), True),
        T.StructField("sample_csv",           T.StringType(), True),
        T.StructField("sample_fetch_status",  T.StringType(), True),
        T.StructField("error_message",        T.StringType(), True),
    ])


def normalize_output_rows(output_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    required_cols = [
        "dataset_id", "source", "title", "original_description",
        "download_url", "landing_page_url", "sample_csv",
        "sample_fetch_status", "error_message",
    ]
    out_pd = pd.DataFrame(output_rows)
    for col in required_cols:
        if col not in out_pd.columns:
            out_pd[col] = None
    out_pd = out_pd[required_cols]
    for col in required_cols:
        out_pd[col] = out_pd[col].where(pd.notnull(out_pd[col]), None)
    return out_pd


In [8]:
step1_df = spark.read.parquet(HDFS_METADATA)

metadata_pd = (
    step1_df
    .select("dataset_id", "source", "title", "original_description",
            "download_url", "landing_page_url")
    .toPandas()
)

output_rows: List[Dict[str, Any]] = []
total = len(metadata_pd)
print(f"[INFO] Total metadata rows to process: {total}")

for idx, row in metadata_pd.iterrows():
    dataset_id           = row.get("dataset_id")
    source               = row.get("source")
    title                = row.get("title")
    original_description = row.get("original_description")
    download_url         = row.get("download_url")
    landing_page_url     = row.get("landing_page_url")

    print(f"[INFO] Fetching sample {idx + 1}/{total} for source={source} dataset_id={dataset_id}")

    sample_csv    = None
    error_message = None
    status        = "error"

    try:
        if download_url is None or not str(download_url).strip():
            raise ValueError("missing_download_url")
        sample_csv = fetch_csv_sample_text(
            download_url=str(download_url),
            max_rows=MAX_SAMPLE_ROWS,
            request_timeout=REQUEST_TIMEOUT,
            max_elapsed_seconds=MAX_ELAPSED_SECONDS,
        )
        if sample_csv is None or not str(sample_csv).strip():
            raise ValueError("empty_or_unreadable_sample")
        status = "success"
    except Exception as e:
        error_message = str(e)
        print(f"[ERROR] source={source} dataset_id={dataset_id} error={error_message}")

    output_rows.append({
        "dataset_id":           None if dataset_id is None else str(dataset_id),
        "source":               None if source is None else str(source),
        "title":                None if title is None else str(title),
        "original_description": None if original_description is None else str(original_description),
        "download_url":         None if download_url is None else str(download_url),
        "landing_page_url":     None if landing_page_url is None else str(landing_page_url),
        "sample_csv":           sample_csv,
        "sample_fetch_status":  status,
        "error_message":        error_message,
    })

if not output_rows:
    raise ValueError("No output rows were produced.")

out_pd        = normalize_output_rows(output_rows)
samples_spark = spark.createDataFrame(out_pd, schema=build_samples_schema())
samples_spark.write.mode("overwrite").parquet(HDFS_SAMPLES)

print(f"[INFO] Wrote {len(out_pd)} sampled datasets to {HDFS_SAMPLES}")
samples_spark.groupBy("source", "sample_fetch_status").count().show(truncate=False)
samples_spark.select(
    "dataset_id", "source", "title", "sample_fetch_status", "error_message"
).show(100, truncate=False)


[INFO] Total metadata rows to process: 400
[INFO] Fetching sample 1/400 for source=nyc_open_data dataset_id=qhkz-4dqm
[INFO] Fetching sample 2/400 for source=nyc_open_data dataset_id=wgnh-qwsg
[INFO] Fetching sample 3/400 for source=nyc_open_data dataset_id=naav-ygga
[INFO] Fetching sample 4/400 for source=nyc_open_data dataset_id=5mb3-padx
[INFO] Fetching sample 5/400 for source=nyc_open_data dataset_id=i2im-iqtt
[INFO] Fetching sample 6/400 for source=nyc_open_data dataset_id=gdk4-mbsv
[INFO] Fetching sample 7/400 for source=nyc_open_data dataset_id=pztn-9bne
[INFO] Fetching sample 8/400 for source=nyc_open_data dataset_id=5ucz-vwe8
[INFO] Fetching sample 9/400 for source=nyc_open_data dataset_id=m5vz-tzqv
[INFO] Fetching sample 10/400 for source=nyc_open_data dataset_id=8zf9-spf8
[INFO] Fetching sample 11/400 for source=nyc_open_data dataset_id=wh8n-imgd
[INFO] Fetching sample 12/400 for source=nyc_open_data dataset_id=ct66-47at
[INFO] Fetching sample 13/400 for source=nyc_open_data

[INFO] Fetching sample 78/400 for source=nyc_open_data dataset_id=dwrg-kzni
[INFO] Fetching sample 79/400 for source=nyc_open_data dataset_id=kwss-yksz
[INFO] Fetching sample 80/400 for source=nyc_open_data dataset_id=wyj6-frpa
[INFO] Fetching sample 81/400 for source=nyc_open_data dataset_id=kizp-4dfk
[INFO] Fetching sample 82/400 for source=nyc_open_data dataset_id=2c5m-rke8
[ERROR] source=nyc_open_data dataset_id=2c5m-rke8 error=empty_or_unreadable_sample
[INFO] Fetching sample 83/400 for source=nyc_open_data dataset_id=2w2g-fk3i
[INFO] Fetching sample 84/400 for source=nyc_open_data dataset_id=2juy-aj8e
[INFO] Fetching sample 85/400 for source=nyc_open_data dataset_id=yqww-f9f3
[ERROR] source=nyc_open_data dataset_id=yqww-f9f3 error=empty_or_unreadable_sample
[INFO] Fetching sample 86/400 for source=nyc_open_data dataset_id=53n2-m85m
[ERROR] source=nyc_open_data dataset_id=53n2-m85m error=empty_or_unreadable_sample
[INFO] Fetching sample 87/400 for source=nyc_open_data dataset_id=h

[INFO] Fetching sample 145/400 for source=nyc_open_data dataset_id=mhst-xhix
[ERROR] source=nyc_open_data dataset_id=mhst-xhix error=empty_or_unreadable_sample
[INFO] Fetching sample 146/400 for source=nyc_open_data dataset_id=ce23-cck4
[INFO] Fetching sample 147/400 for source=nyc_open_data dataset_id=484j-8mzq
[ERROR] source=nyc_open_data dataset_id=484j-8mzq error=empty_or_unreadable_sample
[INFO] Fetching sample 148/400 for source=nyc_open_data dataset_id=bhci-bpwh
[INFO] Fetching sample 149/400 for source=nyc_open_data dataset_id=4gme-r92c
[ERROR] source=nyc_open_data dataset_id=4gme-r92c error=empty_or_unreadable_sample
[INFO] Fetching sample 150/400 for source=nyc_open_data dataset_id=waa5-kewj
[ERROR] source=nyc_open_data dataset_id=waa5-kewj error=empty_or_unreadable_sample
[INFO] Fetching sample 151/400 for source=nyc_open_data dataset_id=45at-qem6
[INFO] Fetching sample 152/400 for source=nyc_open_data dataset_id=esve-a9r5
[ERROR] source=nyc_open_data dataset_id=esve-a9r5 er

[WARN] sample fetch failed attempt=2/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=3/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] giving up on url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] source=data_gov dataset_id=dd80845b-924e-40d3-964c-dd4875113f0a error=empty_or_unreadable_sample
[INFO] Fetching sample 210/400 for source=data_gov dataset_id=015-USMint-0005
[WARN] sample fetch failed attempt=1/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=2/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=3/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] giving up on url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] source=data_gov dat

[INFO] Fetching sample 229/400 for source=data_gov dataset_id=https://data.wa.gov/api/views/4wur-kfnr
[WARN] sample fetch failed attempt=1/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=2/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=3/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] giving up on url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] source=data_gov dataset_id=https://data.wa.gov/api/views/4wur-kfnr error=empty_or_unreadable_sample
[INFO] Fetching sample 230/400 for source=data_gov dataset_id=https://data.montgomerycountymd.gov/api/views/v76h-r7br
[INFO] Fetching sample 231/400 for source=data_gov dataset_id=547
[WARN] sample fetch failed attempt=1/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt

...

[INFO] Fetching sample 388/400 for source=data_gov dataset_id=HUD031
[WARN] sample fetch failed attempt=1/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=2/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=3/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] giving up on url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[ERROR] source=data_gov dataset_id=HUD031 error=empty_or_unreadable_sample
[INFO] Fetching sample 389/400 for source=data_gov dataset_id=https://data.ny.gov/api/views/fgm6-ccue
[WARN] sample fetch failed attempt=1/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=2/3 url=nan: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
[WARN] sample fetch failed attempt=3/3 url=nan: Invalid URL 'nan

26/05/02 22:21:22 WARN TaskSetManager: Stage 14 contains a task of very large size (32985 KiB). The maximum recommended task size is 1000 KiB.


[INFO] Wrote 400 sampled datasets to hdfs:///user/rbp5812_nyu_edu/pipeline/step2_samples


26/05/02 22:21:23 WARN TaskSetManager: Stage 15 contains a task of very large size (32985 KiB). The maximum recommended task size is 1000 KiB.
26/05/02 22:21:25 WARN TaskSetManager: Stage 18 contains a task of very large size (32985 KiB). The maximum recommended task size is 1000 KiB.


+-------------+-------------------+-----+
|source       |sample_fetch_status|count|
+-------------+-------------------+-----+
|nyc_open_data|success            |162  |
|nyc_open_data|error              |38   |
|data_gov     |success            |70   |
|data_gov     |error              |130  |
+-------------+-------------------+-----+

+----------+-------------+--------------------------------------------------------------------------------------------------------+-------------------+--------------------------+
|dataset_id|source       |title                                                                                                   |sample_fetch_status|error_message             |
+----------+-------------+--------------------------------------------------------------------------------------------------------+-------------------+--------------------------+
|qhkz-4dqm |nyc_open_data|Citywide Mobility Survey - Vehicle 2024                                                             

---
## Step 6 — Data Preparation & Filtering

We filter the sampled dataset to keep only records where:
1. `sample_fetch_status == "success"` — CSV was successfully downloaded
2. `sample_csv` is not null or empty

This produces a clean Parquet file at `step3_prepared` containing only datasets ready for LLM description generation.

In [9]:
step2_df = spark.read.parquet(HDFS_SAMPLES)

prepared_df = (
    step2_df
    .filter(F.col("sample_fetch_status") == "success")
    .filter(F.col("sample_csv").isNotNull())
    .filter(F.trim(F.col("sample_csv")) != "")
    .select(
        F.col("dataset_id").cast("string").alias("dataset_id"),
        F.col("title").cast("string").alias("title"),
        F.col("source").cast("string").alias("source"),
        F.col("sample_csv").cast("string").alias("sample_csv"),
        F.col("original_description").cast("string").alias("original_description"),
        F.col("download_url").cast("string").alias("download_url"),
        F.col("landing_page_url").cast("string").alias("landing_page_url"),
    )
)

prepared_df.write.mode("overwrite").parquet(HDFS_PREPARED)

print("[INFO] Prepared records written.")
print(f"[INFO] Output path: {HDFS_PREPARED}")
print(f"[INFO] Prepared row count: {prepared_df.count()}")
prepared_df.groupBy("source").count().show(truncate=False)
prepared_df.select("dataset_id", "source", "title").show(100, truncate=False)


[INFO] Prepared records written.
[INFO] Output path: hdfs:///user/rbp5812_nyu_edu/pipeline/step3_prepared


[INFO] Prepared row count: 232
+-------------+-----+
|source       |count|
+-------------+-----+
|data_gov     |70   |
|nyc_open_data|162  |
+-------------+-----+

+----------+-------------+--------------------------------------------------------------------------------------------------------+
|dataset_id|source       |title                                                                                                   |
+----------+-------------+--------------------------------------------------------------------------------------------------------+
|qhkz-4dqm |nyc_open_data|Citywide Mobility Survey - Vehicle 2024                                                                 |
|wgnh-qwsg |nyc_open_data|Citywide Mobility Survey - Trip 2024                                                                    |
|naav-ygga |nyc_open_data|Citywide Mobility Survey - Day 2024                                                                     |
|5mb3-padx |nyc_open_data|Citywide Mobility 

### Helper Functions
Utility functions for copying HDFS parquet files to local storage and truncating large CSV samples before sending to the LLM.

In [10]:
def run_cmd(cmd: List[str]) -> None:
    print(f"[CMD] {' '.join(cmd)}")
    subprocess.run(cmd, check=True)


def shrink_sample_csv(
    sample_csv: Optional[str],
    max_chars: int = 8000,
    max_lines: int = 8,
) -> Optional[str]:
    if sample_csv is None:
        return None
    text = str(sample_csv).strip()
    if not text:
        return None
    lines      = text.splitlines()
    if not lines:
        return None
    header     = lines[0]
    data_lines = lines[1 : 1 + max_lines]
    shrunk     = "\n".join([header] + data_lines)
    if len(shrunk) > max_chars:
        shrunk = shrunk[:max_chars]
    return shrunk.strip() if shrunk.strip() else None


def copy_hdfs_parquet_to_local(hdfs_path: str, local_parent_dir: str) -> str:
    local_path = os.path.join(local_parent_dir, Path(hdfs_path).name)
    if os.path.exists(local_path):
        shutil.rmtree(local_path, ignore_errors=True)
    run_cmd(["hdfs", "dfs", "-get", hdfs_path, local_parent_dir])
    return local_path


---
## Step 7 — LLM Description Generation

For each prepared dataset we:
1. **Shrink** the sample CSV to fit within `MAX_CHARS` / `MAX_LINES` limits before sending to the LLM
2. **Call** `AutoDDG.describe_dataset()` which sends the sample to GPT-4o-mini with a structured prompt
3. **Store** the generated description along with generation status and any error messages

The OpenAI API key must be set via `os.environ["OPENAI_API_KEY"]` before running this cell.

Results are saved locally as Parquet at `LOCAL_OUTPUT_PARQUET` then uploaded to HDFS for Spark to read in the next step.

> **Cost estimate**: ~400 datasets × 500 tokens ≈ 200K tokens ≈ **< $0.05** with gpt-4o-mini

In [11]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE"

In [12]:
with tempfile.TemporaryDirectory() as tmpdir:
    local_input_parquet = copy_hdfs_parquet_to_local(HDFS_PREPARED, tmpdir)
    records_pd = pd.read_parquet(local_input_parquet)

    client  = OpenAI()
    autoddg = AutoDDG(client=client, model_name=MODEL_NAME)

    output_rows: List[Dict[str, Any]] = []
    total = len(records_pd)
    print(f"[INFO] Total prepared records to describe: {total}")

    for idx, row in records_pd.iterrows():
        dataset_id           = row.get("dataset_id")
        source               = row.get("source")
        title                = row.get("title")
        original_description = row.get("original_description")
        download_url         = row.get("download_url")
        landing_page_url     = row.get("landing_page_url")
        sample_csv           = row.get("sample_csv")

        print(f"[INFO] Generating description {idx + 1}/{total} for source={source} dataset_id={dataset_id}")

        generated_description = None
        generation_status     = "error"
        error_message         = None

        try:
            shrunk_sample_csv = shrink_sample_csv(
                sample_csv=sample_csv,
                max_chars=MAX_CHARS,
                max_lines=MAX_LINES,
            )
            if shrunk_sample_csv is None:
                raise ValueError("missing_or_empty_sample_csv_after_shrink")
            _, generated_description = autoddg.describe_dataset(
                dataset_sample=shrunk_sample_csv
            )
            generation_status = "success"
        except Exception as e:
            error_message = str(e)
            print(f"[ERROR] source={source} dataset_id={dataset_id} error={error_message}")

        output_rows.append({
            "dataset_id":            None if pd.isna(dataset_id) else str(dataset_id),
            "source":                None if pd.isna(source) else str(source),
            "title":                 None if pd.isna(title) else str(title),
            "original_description":  None if pd.isna(original_description) else str(original_description),
            "download_url":          None if pd.isna(download_url) else str(download_url),
            "landing_page_url":      None if pd.isna(landing_page_url) else str(landing_page_url),
            "sample_csv":            None if pd.isna(sample_csv) else str(sample_csv),
            "generated_description": generated_description,
            "generation_status":     generation_status,
            "error_message":         error_message,
        })

out_df      = pd.DataFrame(output_rows)
output_path = Path(LOCAL_OUTPUT_PARQUET)
output_path.parent.mkdir(parents=True, exist_ok=True)
out_df.to_parquet(output_path, index=False)

print(f"[INFO] Wrote {len(out_df)} rows locally to {output_path}")
print(out_df.groupby(["source", "generation_status"]).size().reset_index(name="count"))


[CMD] hdfs dfs -get hdfs:///user/rbp5812_nyu_edu/pipeline/step3_prepared /tmp/tmptxbk3316
[INFO] Total prepared records to describe: 232
[INFO] Generating description 1/232 for source=nyc_open_data dataset_id=qhkz-4dqm
[INFO] Generating description 2/232 for source=nyc_open_data dataset_id=wgnh-qwsg
[INFO] Generating description 3/232 for source=nyc_open_data dataset_id=naav-ygga
[INFO] Generating description 4/232 for source=nyc_open_data dataset_id=5mb3-padx
[INFO] Generating description 5/232 for source=nyc_open_data dataset_id=i2im-iqtt
[INFO] Generating description 6/232 for source=nyc_open_data dataset_id=gdk4-mbsv
[INFO] Generating description 7/232 for source=nyc_open_data dataset_id=pztn-9bne
[INFO] Generating description 8/232 for source=nyc_open_data dataset_id=5ucz-vwe8
[INFO] Generating description 9/232 for source=nyc_open_data dataset_id=m5vz-tzqv
[INFO] Generating description 10/232 for source=nyc_open_data dataset_id=8zf9-spf8
[INFO] Generating description 11/232 for s

[INFO] Generating description 99/232 for source=nyc_open_data dataset_id=v6kb-cqej
[INFO] Generating description 100/232 for source=nyc_open_data dataset_id=4kc9-zrs2
[INFO] Generating description 101/232 for source=nyc_open_data dataset_id=jqfp-uff7
[INFO] Generating description 102/232 for source=nyc_open_data dataset_id=5mad-ntua
[INFO] Generating description 103/232 for source=nyc_open_data dataset_id=if4c-w48d
[INFO] Generating description 104/232 for source=nyc_open_data dataset_id=eymt-yinc
[INFO] Generating description 105/232 for source=nyc_open_data dataset_id=qzji-nvbd
[INFO] Generating description 106/232 for source=nyc_open_data dataset_id=36nr-7fbp
[INFO] Generating description 107/232 for source=nyc_open_data dataset_id=2i8t-es4u
[INFO] Generating description 108/232 for source=nyc_open_data dataset_id=83yx-cgpy
[INFO] Generating description 109/232 for source=nyc_open_data dataset_id=bqye-aqft
[INFO] Generating description 110/232 for source=nyc_open_data dataset_id=tja

[INFO] Generating description 188/232 for source=data_gov dataset_id=https://data.cdc.gov/api/views/55yu-xksw
[INFO] Generating description 189/232 for source=data_gov dataset_id=9e9ce3da-64a6-431a-b1ec-a4ba001a2c53
[INFO] Generating description 190/232 for source=data_gov dataset_id=https://data.cityofnewyork.us/api/views/uip8-fykc
[INFO] Generating description 191/232 for source=data_gov dataset_id=https://data.cityofnewyork.us/api/views/25th-nujf
[INFO] Generating description 192/232 for source=data_gov dataset_id=https://data.cdc.gov/api/views/w9j2-ggv5
[INFO] Generating description 193/232 for source=data_gov dataset_id=https://data.kingcounty.gov/api/views/ngy6-yfbq
[INFO] Generating description 194/232 for source=data_gov dataset_id=https://data.cityofchicago.org/api/views/85ca-t3if
[INFO] Generating description 195/232 for source=data_gov dataset_id=https://doi.org/10.23719/1528686
[INFO] Generating description 196/232 for source=data_gov dataset_id=https://data.cityofchicago.o

---
## Step 8 — Evaluation Table Construction

We read the generated descriptions and enrich them with:
- **`has_original_description`**: whether the dataset had a human-written description to compare against
- **`original_description_len`**: character length of the original description
- **`generated_description_len`**: character length of the generated description

This evaluation table is written to HDFS at `step5_evaluation` for analysis.

In [13]:
# First copy local parquet to HDFS
import subprocess
subprocess.run([
    "hdfs", "dfs", "-put", "-f",
    LOCAL_OUTPUT_PARQUET,
    f"{HDFS_BASE}/step4_descriptions.parquet"
], check=True)

# Now read from HDFS
desc_df = spark.read.parquet(f"{HDFS_BASE}/step4_descriptions.parquet")

eval_df = (
    desc_df
    .withColumn(
        "has_original_description",
        F.when(
            F.col("original_description").isNotNull() &
            (F.trim(F.col("original_description")) != ""),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "original_description_len",
        F.length(F.coalesce(F.col("original_description"), F.lit("")))
    )
    .withColumn(
        "generated_description_len",
        F.length(F.coalesce(F.col("generated_description"), F.lit("")))
    )
    .select(
        "dataset_id", "source", "title",
        "has_original_description",
        "original_description", "generated_description",
        "original_description_len", "generated_description_len",
        "generation_status", "error_message",
    )
)

eval_df.write.mode("overwrite").parquet(HDFS_EVAL)

print("[INFO] Evaluation table written.")
print(f"[INFO] Output path: {HDFS_EVAL}")
print(f"[INFO] Row count: {eval_df.count()}")
eval_df.groupBy("source", "generation_status").count().show(truncate=False)
eval_df.groupBy("source", "has_original_description").count().show(truncate=False)
eval_df.select(
    "dataset_id", "source", "title",
    "original_description_len", "generated_description_len",
).show(50, truncate=False)

[INFO] Evaluation table written.
[INFO] Output path: hdfs:///user/rbp5812_nyu_edu/pipeline/step5_evaluation
[INFO] Row count: 232
+-------------+-----------------+-----+
|source       |generation_status|count|
+-------------+-----------------+-----+
|nyc_open_data|success          |162  |
|data_gov     |success          |70   |
+-------------+-----------------+-----+

+-------------+------------------------+-----+
|source       |has_original_description|count|
+-------------+------------------------+-----+
|nyc_open_data|false                   |5    |
|data_gov     |true                    |70   |
|nyc_open_data|true                    |157  |
+-------------+------------------------+-----+

+----------+-------------+---------------------------------------------------------------------------------------------------+------------------------+-------------------------+
|dataset_id|source       |title                                                                                          

In [14]:
results_pd = eval_df.toPandas()
successful = results_pd[results_pd["generation_status"] == "success"]

print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"Datasets requested           : {NYC_LIMIT + DATA_GOV_LIMIT}")
print(f"  NYC Open Data              : {NYC_LIMIT}")
print(f"  Data.gov                   : {DATA_GOV_LIMIT}")
print(f"Descriptions generated       : {len(successful)}")
print(f"Generation errors            : {len(results_pd) - len(successful)}")
print()
print("Generated description length stats:")
print(successful["generated_description_len"].describe().round(1).to_string())
print("=" * 60)


PIPELINE SUMMARY
Datasets requested           : 400
  NYC Open Data              : 200
  Data.gov                   : 200
Descriptions generated       : 232
Generation errors            : 0

Generated description length stats:
count    232.0
mean     640.2
std       40.7
min      529.0
25%      609.8
50%      640.0
75%      666.2
max      785.0


In [15]:
sample_n = min(5, len(successful))
for _, row in successful.sample(n=sample_n, random_state=42).iterrows():
    print("─" * 60)
    print(f"Dataset : {row['title']}")
    print(f"Source  : {row['source']}")
    print()
    if row.get("original_description"):
        print("Original description:")
        print(f"  {str(row['original_description'])[:300]}")
        print()
    print("Generated description:")
    print(f"  {row['generated_description']}")
    print()


────────────────────────────────────────────────────────────
Dataset : Intercity Bus Stop Permits
Source  : data_gov

Original description:
  Intercity bus operators must obtain a permit from DOT before they can make on-street stops in the city.

Generated description:
  This dataset contains information about intercity bus stop permits issued in New York City. Each entry includes details such as the permit number, application type, status, issue dates, and the specific location of the bus stop, including street names and boroughs. The dataset also outlines stipulations for each permit, emphasizing that only one bus is allowed at a time and that permit holders must provide an approved schedule upon request. The dataset features various companies, including Hampton Jitney, Inc. and Transdev Services, Inc., and includes both active and expired permits.

────────────────────────────────────────────────────────────
Dataset : 311 Customer Satisfaction Survey
Source  : nyc_open_data

Origina

---
## Step 9 — Evaluation: Text Similarity Metrics

We evaluate the quality of generated descriptions against original human-written descriptions using three complementary metrics:

| Metric | What it measures |
|--------|-----------------|
| **ROUGE-1** | Unigram (word) overlap between generated and original |
| **ROUGE-2** | Bigram overlap — captures phrase-level similarity |
| **ROUGE-L** | Longest common subsequence — measures fluency |
| **METEOR** | Word overlap accounting for synonyms and stemming |
| **BERTScore** | Deep semantic similarity using contextual embeddings |

Only datasets that have **both** an original and a generated description are included in this comparison (`n = 227`).

In [16]:
# Install evaluation libraries
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", 
                       "rouge-score", "bert-score", "nltk", "--quiet"])

0

In [17]:
import pandas as pd
from rouge_score import rouge_scorer
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.meteor_score import meteor_score
import numpy as np

# Load results
results_pd = pd.read_parquet("/home/rbp5812_nyu_edu/outputs/descriptions.parquet")

comparable = results_pd[
    results_pd["original_description"].notna() &
    results_pd["generated_description"].notna() &
    (results_pd["original_description"].str.strip() != "") &
    (results_pd["generation_status"] == "success")
].copy()

print(f"Datasets with both original and generated: {len(comparable)}")

# ROUGE
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
rouge1_scores, rouge2_scores, rougeL_scores = [], [], []
for _, row in comparable.iterrows():
    s = scorer.score(row["original_description"], row["generated_description"])
    rouge1_scores.append(s["rouge1"].fmeasure)
    rouge2_scores.append(s["rouge2"].fmeasure)
    rougeL_scores.append(s["rougeL"].fmeasure)

print("\n── ROUGE Scores ──────────────────────────")
print(f"ROUGE-1 : {np.mean(rouge1_scores):.4f}")
print(f"ROUGE-2 : {np.mean(rouge2_scores):.4f}")
print(f"ROUGE-L : {np.mean(rougeL_scores):.4f}")

# METEOR
meteor_scores = []
for _, row in comparable.iterrows():
    ref = row["original_description"].split()
    hyp = row["generated_description"].split()
    meteor_scores.append(meteor_score([ref], hyp))

print("\n── METEOR Score ──────────────────────────")
print(f"METEOR  : {np.mean(meteor_scores):.4f}")

# Store scores
comparable["rouge1"] = rouge1_scores
comparable["rouge2"] = rouge2_scores
comparable["rougeL"] = rougeL_scores
comparable["meteor"] = meteor_scores

print("\n── Per-source breakdown (ROUGE + METEOR) ─")
print(comparable.groupby("source")[["rouge1","rouge2","rougeL","meteor"]].mean().round(4))

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/rbp5812_nyu_edu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/rbp5812_nyu_edu/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Datasets with both original and generated: 227

── ROUGE Scores ──────────────────────────
ROUGE-1 : 0.2437
ROUGE-2 : 0.0450
ROUGE-L : 0.1430

── METEOR Score ──────────────────────────
METEOR  : 0.1593

── Per-source breakdown (ROUGE + METEOR) ─
               rouge1  rouge2  rougeL  meteor
source                                       
data_gov       0.2609  0.0597  0.1539  0.1610
nyc_open_data  0.2361  0.0384  0.1381  0.1586


---
## Step 10 — Evaluation: Retrieval (NDCG@10)

We simulate a dataset search engine scenario. For each test query we check how many relevant keywords appear in each description, then compute **NDCG@10** (Normalized Discounted Cumulative Gain).

A higher NDCG means relevant datasets appear earlier in search results. We compare:
- **Generated** descriptions (our AutoDDG output)
- **Original** descriptions (human-written baseline)

Test queries are chosen to match known datasets in our collection.

In [18]:
import numpy as np
import pandas as pd

results_pd = pd.read_parquet("/home/rbp5812_nyu_edu/outputs/descriptions.parquet")

# Define test queries and relevant dataset keywords
test_queries = {
    "shooting incidents NYC"      : ["shooting", "offender", "victim", "crime"],
    "NYC parking violations"      : ["parking", "violation", "fiscal"],
    "flood sensor data"           : ["flood", "sensor", "floodnet"],
    "bicycle pedestrian counts"   : ["bicycle", "pedestrian", "count"],
    "COVID hospitalization rates" : ["covid", "hospitalization", "monthly"],
    "electric vehicle population" : ["electric", "vehicle", "population"],
    "chronic disease indicators"  : ["chronic", "disease", "behavioral"],
    "motor vehicle collisions"    : ["collision", "crash", "vehicle"],
}

def compute_ndcg(query_keywords, descriptions, k=10):
    scores = []
    for desc in descriptions:
        desc_lower = str(desc).lower()
        hit = sum(1 for kw in query_keywords if kw in desc_lower)
        scores.append(hit)
    ideal  = sorted(scores, reverse=True)[:k]
    actual = scores[:k]
    def dcg(rels):
        return sum(rel / np.log2(idx + 2) for idx, rel in enumerate(rels))
    idcg = dcg(ideal)
    return dcg(actual) / idcg if idcg > 0 else 0.0

successful = results_pd[results_pd["generation_status"] == "success"]
all_generated  = successful["generated_description"].tolist()
all_original   = results_pd[results_pd["original_description"].notna()]["original_description"].tolist()

print("── Retrieval Evaluation (NDCG@10) ───────────────────────────")
print(f"{'Query':<40} {'Generated':>10} {'Original':>10}")
print("-" * 62)

ndcg_gen, ndcg_orig = [], []
for query, keywords in test_queries.items():
    gen_ndcg  = compute_ndcg(keywords, all_generated)
    orig_ndcg = compute_ndcg(keywords, all_original)
    ndcg_gen.append(gen_ndcg)
    ndcg_orig.append(orig_ndcg)
    print(f"{query:<40} {gen_ndcg:>10.4f} {orig_ndcg:>10.4f}")

print("-" * 62)
print(f"{'Average':<40} {np.mean(ndcg_gen):>10.4f} {np.mean(ndcg_orig):>10.4f}")

── Retrieval Evaluation (NDCG@10) ───────────────────────────
Query                                     Generated   Original
--------------------------------------------------------------
shooting incidents NYC                       0.1382     0.3133
NYC parking violations                       0.2933     0.3076
flood sensor data                            0.0000     0.0000
bicycle pedestrian counts                    0.0777     0.0454
COVID hospitalization rates                  0.0000     0.0000
electric vehicle population                  0.3995     0.4418
chronic disease indicators                   0.0000     0.0000
motor vehicle collisions                     0.3137     0.3638
--------------------------------------------------------------
Average                                      0.1528     0.1840


---
## Step 11 — Evaluation: BERTScore

BERTScore measures semantic similarity using contextual embeddings from a pretrained language model (`distilbert-base-uncased`). Unlike ROUGE/METEOR which rely on exact word matches, BERTScore captures meaning even when different words are used.

We use `batch_size=4` to avoid out-of-memory errors on the master node.

In [19]:
from bert_score import score as bert_score
import numpy as np

refs = comparable["original_description"].tolist()
hyps = comparable["generated_description"].tolist()

P, R, F1 = bert_score(hyps, refs, lang="en",
                       model_type="distilbert-base-uncased",
                       batch_size=4,   # small batch to save memory
                       verbose=True)

print("\n── BERTScore (distilbert) ────────────────")
print(f"Precision : {P.mean().item():.4f}")
print(f"Recall    : {R.mean().item():.4f}")
print(f"F1        : {F1.mean().item():.4f}")

comparable["bertscore_f1"] = F1.numpy()
print("\n── Full per-source breakdown ─────────────")
print(comparable.groupby("source")[["rouge1","rouge2","rougeL","meteor","bertscore_f1"]].mean().round(4))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/113 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/57 [00:00<?, ?it/s]

done in 133.80 seconds, 1.70 sentences/sec

── BERTScore (distilbert) ────────────────
Precision : 0.7565
Recall    : 0.7185
F1        : 0.7350

── Full per-source breakdown ─────────────
               rouge1  rouge2  rougeL  meteor  bertscore_f1
source                                                     
data_gov       0.2609  0.0597  0.1539  0.1610        0.7526
nyc_open_data  0.2361  0.0384  0.1381  0.1586        0.7272


---
## Step 12 — Qualitative Evaluation

We randomly sample 5 successfully described datasets and display both the original and generated descriptions side by side for manual review.

Key aspects to assess:
- **Accuracy** — does the generated description correctly reflect the dataset content?
- **Readability** — is it clear and well-structured?
- **Informativeness** — does it add value over the original?

In [20]:
import random
random.seed(42)

successful = results_pd[results_pd["generation_status"] == "success"].copy()
sample = successful.sample(n=min(5, len(successful)), random_state=42)

print("=" * 70)
print("QUALITATIVE EVALUATION SAMPLE")
print("=" * 70)

for _, row in sample.iterrows():
    print(f"\nDataset : {row['title']}")
    print(f"Source  : {row['source']}")
    print(f"─" * 70)
    
    if pd.notna(row.get("original_description")) and str(row["original_description"]).strip():
        orig = str(row["original_description"])[:400]
        print(f"Original Description:\n  {orig}...")
    else:
        print("Original Description: [None]")
    
    print(f"\nGenerated Description:\n  {row['generated_description']}")
    print(f"\nGenerated Length : {len(str(row['generated_description']))} chars")
    print("=" * 70)

QUALITATIVE EVALUATION SAMPLE

Dataset : Intercity Bus Stop Permits
Source  : data_gov
──────────────────────────────────────────────────────────────────────
Original Description:
  Intercity bus operators must obtain a permit from DOT before they can make on-street stops in the city....

Generated Description:
  This dataset contains information about intercity bus stop permits issued in New York City. Each entry includes details such as the permit number, application type, status, issue dates, and the specific location of the bus stop, including street names and boroughs. The dataset also outlines stipulations for each permit, emphasizing that only one bus is allowed at a time and that permit holders must provide an approved schedule upon request. The dataset features various companies, including Hampton Jitney, Inc. and Transdev Services, Inc., and includes both active and expired permits.

Generated Length : 590 chars

Dataset : 311 Customer Satisfaction Survey
Source  : nyc_open_d

---
## Pipeline Complete

All steps have been executed successfully. Below is a summary of outputs:

| Output | Location |
|--------|----------|
| Metadata | `hdfs:///user/rbp5812_nyu_edu/pipeline/step1_metadata` |
| CSV Samples | `hdfs:///user/rbp5812_nyu_edu/pipeline/step2_samples` |
| Prepared Data | `hdfs:///user/rbp5812_nyu_edu/pipeline/step3_prepared` |
| Generated Descriptions | `/home/rbp5812_nyu_edu/outputs/descriptions.parquet` |
| Evaluation Table | `hdfs:///user/rbp5812_nyu_edu/pipeline/step5_evaluation` |

Stopping the Spark session releases cluster resources.

In [21]:
spark.stop()
print("Spark session stopped.")
print(f"Final descriptions : {LOCAL_OUTPUT_PARQUET}")
print(f"Evaluation table   : {HDFS_EVAL}")


Spark session stopped.
Final descriptions : /home/rbp5812_nyu_edu/outputs/descriptions.parquet
Evaluation table   : hdfs:///user/rbp5812_nyu_edu/pipeline/step5_evaluation
